## Questão 4 - Análise de clientes

### Cenário

A Diretoria da LH Nautical deseja identificar os clientes fieis. Diferente de quem compra muito uma única vez, o cliente fiel é o cliente que possui um gasto médio alto por transação e navega por diversas categorias da loja. O objetivo é mapear o que esses clientes de elite estão consumindo para replicar o comportamento em outros segmentos.

### Premissas obrigatórias:
- Faturamento Total: Soma da coluna total por cliente.
- Frequência: Contagem total de transações (IDs de venda) por cliente.
- Ticket Médio: Faturamento Total / Frequência.
- Diversidade de Categorias: Quantidade de categorias distintas (category_id) que o cliente comprou.
- Filtro de Elite: Apenas clientes que compraram produtos de 13 ou mais categorias distintas devem ser considerados no ranking.
- Desempate: Em caso de empate no Ticket Médio, utilize o customer_id em ordem crescente.

### Tarefa:
1. Calcule o Ticket Médio e a Diversidade de Categorias para cada customer_id.
2. Filtre os 10 clientes com o maior Ticket Médio que atendam ao critério de diversidade (13 ou + categorias).
3. Para este grupo específico de 10 clientes, identifique qual categoria de produto concentra a maior quantidade total de itens comprados (sum(quantity)).


## 1. Ticket Médio e Diversidade de Categorias para cada `customer_id`

Nesta etapa, foram calculados indicadores de comportamento de compra para cada cliente da base.

O objetivo foi reunir, em uma única consulta, informações sobre o valor movimentado, a frequência de pedidos, o ticket médio e a variedade de categorias adquiridas por cada `customer_id`.

Para isso, os indicadores financeiros foram calculados diretamente na tabela `orders`, enquanto a diversidade de categorias foi obtida por meio do relacionamento entre pedidos, itens, variações e produtos.

As métricas resultantes permitem compreender não apenas quanto cada cliente movimentou, mas também a recorrência e a diversidade de suas compras.



## Métricas utilizadas

A análise foi dividida em duas etapas para evitar duplicidade de valores.

Na primeira etapa, os indicadores financeiros e de frequência foram calculados diretamente na tabela `orders`. Dessa forma, cada pedido foi considerado apenas uma vez.

Na segunda etapa, foram utilizados os relacionamentos entre pedidos, itens, variações e produtos para identificar quantas categorias distintas foram adquiridas por cada cliente.

### Faturamento total

**Fórmula:** `SUM(total)`

Representa a soma do valor total de todos os pedidos associados a cada `customer_id`.

### Frequência de pedidos

**Fórmula:** `COUNT(id)`

Representa a quantidade de pedidos realizados por cada cliente. A contagem foi feita diretamente na tabela `orders`, em que cada `id` representa um pedido.

### Ticket médio

**Fórmula:** `SUM(total) / COUNT(id)`

O ticket médio representa o valor médio movimentado em cada pedido. A função `NULLIF(COUNT(id), 0)` foi utilizada para impedir uma divisão por zero caso não existissem pedidos para um cliente.

### Diversidade de categorias

**Fórmula:** `COUNT(DISTINCT category_id)`

Representa a quantidade de categorias diferentes compradas por cada cliente.

Para chegar à categoria, foi realizado o encadeamento de chaves:

`orders → order_items → product_variants → products`

A função `COUNT(DISTINCT ...)` foi necessária porque um mesmo cliente pode comprar diversos produtos pertencentes à mesma categoria. Portanto, cada categoria foi contabilizada apenas uma vez.

### Integração dos resultados

Por fim, as métricas de pedidos foram unidas à diversidade de categorias pelo campo `customer_id`.

O `LEFT JOIN` garante que todos os clientes presentes em `orders` sejam mantidos no resultado, inclusive se não houver uma categoria associada. Nesses casos, `COALESCE(..., 0)` apresenta a diversidade como zero.

-- ============================================================
-- ETAPA 1 - Faturamento, frequência e ticket médio por cliente
-- ============================================================
```sql 
WITH metricas_pedidos AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / NULLIF(COUNT(id), 0) AS ticket_medio
    FROM orders
    GROUP BY customer_id
),

-- ============================================================
-- ETAPA 2 - Diversidade de categorias por cliente
-- ============================================================
diversidade_categorias AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    INNER JOIN product_variants AS pv
        ON oi.product_variant_id = pv.id
    INNER JOIN products AS p
        ON pv.product_id = p.id
    GROUP BY o.customer_id
)

-- ============================================================
-- ETAPA 3 - Resultado para cada customer_id
-- ============================================================
SELECT
    mp.customer_id,
    mp.faturamento_total,
    mp.frequencia,
    mp.ticket_medio,
    COALESCE(dc.diversidade_categorias, 0) AS diversidade_categorias
FROM metricas_pedidos AS mp
LEFT JOIN diversidade_categorias AS dc
    ON mp.customer_id = dc.customer_id
ORDER BY mp.customer_id;


## 2 - Filtre os 10 clientes com o maior Ticket Médio que atendam ao critério de diversidade (13 ou + categorias)

Com as métricas calculadas na etapa anterior, foi aplicado o critério de diversidade mínima para identificar clientes com comportamento de compra mais variado.

Foram mantidos apenas os clientes com **13 ou mais categorias distintas**, conforme solicitado no desafio.

Após esse filtro, os resultados foram ordenados de forma decrescente pelo ticket médio e limitados aos 10 primeiros registros.

### Critério de diversidade

**Fórmula:** `diversidade_categorias >= 13`

Esse filtro evita que clientes com ticket médio elevado, mas compras concentradas em poucas categorias, sejam classificados como clientes fiéis.

### Critério de ordenação

**Fórmula:** `ORDER BY ticket_medio DESC`

O ticket médio foi utilizado como principal critério de classificação, pois representa o valor médio movimentado por pedido.

Como critério de desempate, foi utilizado o `customer_id` em ordem crescente. Isso torna o resultado reproduzível caso dois clientes tenham o mesmo ticket médio.

### Limite de clientes

**Fórmula:** `LIMIT 10`

Após aplicar o filtro de diversidade e a ordenação pelo ticket médio, foram selecionados os 10 primeiros clientes.

```sql
WITH metricas_pedidos AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / NULLIF(COUNT(id), 0) AS ticket_medio
    FROM orders
    GROUP BY customer_id
),

diversidade_categorias AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    INNER JOIN product_variants AS pv
        ON oi.product_variant_id = pv.id
    INNER JOIN products AS p
        ON pv.product_id = p.id
    GROUP BY o.customer_id
)

SELECT
    mp.customer_id,
    mp.faturamento_total,
    mp.frequencia,
    mp.ticket_medio,
    dc.diversidade_categorias
FROM metricas_pedidos AS mp
INNER JOIN diversidade_categorias AS dc
    ON mp.customer_id = dc.customer_id
WHERE dc.diversidade_categorias >= 13
ORDER BY
    mp.ticket_medio DESC,
    mp.customer_id ASC
LIMIT 10;

## Resultado - Clientes fiéis

Foram identificados os 10 clientes com maior ticket médio entre aqueles que compraram produtos de, no mínimo, 13 categorias distintas.

| Posição | Cliente | Faturamento total | Frequência | Ticket médio | Categorias distintas |
|---:|---:|---:|---:|---:|---:|
| 1 | 22 | R$ 1.087.838,44 | 26 | R$ 41.839,94 | 14 |
| 2 | 1477 | R$ 916.262,58 | 22 | R$ 41.648,30 | 14 |
| 3 | 929 | R$ 1.082.775,89 | 26 | R$ 41.645,23 | 14 |
| 4 | 1116 | R$ 655.737,20 | 16 | R$ 40.983,58 | 14 |
| 5 | 1691 | R$ 815.471,30 | 20 | R$ 40.773,57 | 14 |
| 6 | 774 | R$ 726.127,99 | 18 | R$ 40.340,44 | 14 |
| 7 | 1470 | R$ 1.040.553,09 | 26 | R$ 40.021,27 | 14 |
| 8 | 1599 | R$ 997.616,46 | 25 | R$ 39.904,66 | 14 |
| 9 | 965 | R$ 677.297,78 | 17 | R$ 39.841,05 | 14 |
| 10 | 1722 | R$ 1.146.455,22 | 29 | R$ 39.532,94 | 14 |

## 3. Para este grupo específico de 10 clientes, identifique qual categoria de produto concentra a maior quantidade total de itens comprados (sum(quantity)).

Nesta etapa, foi analisado o comportamento de compra apenas do grupo dos 10 clientes selecionados pelo maior ticket médio e diversidade mínima de 13 categorias.

O objetivo foi identificar qual categoria concentrou a maior quantidade total de itens adquiridos por esse grupo.

A quantidade foi calculada pela soma do campo `quantity` presente em `order_items`. Diferentemente da contagem de pedidos, essa métrica considera o volume efetivo de unidades compradas.

Para chegar à categoria de cada item, foi utilizado o relacionamento:

`orders → order_items → product_variants → products → categories`

Após restringir a análise aos 10 clientes selecionados, as quantidades foram agrupadas por categoria e ordenadas de forma decrescente. A primeira posição representa a categoria com o maior volume total de itens comprados por esse grupo.

```sql
WITH metricas_pedidos AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / NULLIF(COUNT(id), 0) AS ticket_medio
    FROM orders
    GROUP BY customer_id
),

diversidade_categorias AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.id = oi.order_id
    INNER JOIN product_variants AS pv
        ON oi.product_variant_id = pv.id
    INNER JOIN products AS p
        ON pv.product_id = p.id
    GROUP BY o.customer_id
),

top_10_clientes AS (
    SELECT
        mp.customer_id
    FROM metricas_pedidos AS mp
    INNER JOIN diversidade_categorias AS dc
        ON mp.customer_id = dc.customer_id
    WHERE dc.diversidade_categorias >= 13
    ORDER BY
        mp.ticket_medio DESC,
        mp.customer_id ASC
    LIMIT 10
)

SELECT
    c.name AS categoria,
    SUM(oi.quantity) AS quantidade_total_itens
FROM top_10_clientes AS t
INNER JOIN orders AS o
    ON t.customer_id = o.customer_id
INNER JOIN order_items AS oi
    ON o.id = oi.order_id
INNER JOIN product_variants AS pv
    ON oi.product_variant_id = pv.id
INNER JOIN products AS p
    ON pv.product_id = p.id
INNER JOIN categories AS c
    ON p.category_id = c.id
GROUP BY c.name
ORDER BY quantidade_total_itens DESC
LIMIT 1;

## Questão 4.2 - Explique:

## Como você chegou nas categorias mais vendidas? (mapeamento da cadeia de chaves)

A identificação das categorias mais vendidas foi realizada a partir do relacionamento entre pedidos, itens de pedido, variações de produto, produtos e categorias.

A cadeia de chaves utilizada foi:

`orders.customer_id → orders.id → order_items.order_id → order_items.product_variant_id → product_variants.id → product_variants.product_id → products.id → products.category_id → categories.id`

Inicialmente, cada pedido foi associado aos seus itens por meio de `orders.id = order_items.order_id`. Em seguida, cada item foi relacionado à sua variação de produto, ao produto principal e, por fim, à categoria do produto.

Após esse mapeamento, a coluna `order_items.quantity` foi somada por categoria. Foi identificado que a categoria **Hélices** concentrou a maior quantidade de itens comprados pelos 10 clientes fiéis, com **492 unidades**.


## Qual lógica foi utilizada para filtrar os clientes com diversidade mínima?

A diversidade de categorias foi calculada com a função:

`COUNT(DISTINCT products.category_id)`

Essa lógica contabiliza apenas categorias diferentes compradas por cada cliente, sem repetir uma categoria quando foram adquiridos vários produtos ou itens pertencentes a ela.

Após o cálculo, foram mantidos somente os clientes com diversidade maior ou igual a 13 categorias. Entre esse grupo, os clientes foram ordenados pelo ticket médio em ordem decrescente. Em caso de empate, foi utilizado o `customer_id` em ordem crescente. Os 10 primeiros registros formaram o grupo de clientes fiéis.


## Como foi garantido que a contagem de itens refletisse apenas os Top 10?

A seleção dos clientes fiéis foi criada em uma CTE chamada `clientes_fieis`. Essa CTE contém exclusivamente os 10 clientes com maior ticket médio entre aqueles que atenderam ao critério de diversidade mínima.

A soma de `order_items.quantity` por categoria foi realizada somente após o relacionamento da tabela `orders` com a CTE `clientes_fieis`, utilizando o campo `customer_id`.

Dessa forma, foram excluídas da contagem todas as compras feitas por clientes fora do Top 10. Portanto, as 492 unidades da categoria Hélices representam exclusivamente as compras realizadas pelos clientes selecionados.
